## Alert Sytem

This Nootebook is intended to show the process of creating the alert system using the Image Caption, NER and Sentiment Analysis modules.


In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"
import pandas as pd
import torch
from pathlib import Path
import json
import ast
from pydantic import BaseModel, Field
from typing import Optional, Literal
import sys
#!{sys.executable} -m pip install -U google-genai
from google import genai
import time
from utils.ner_functions import normalize_ner_output
from utils.alert_functions import benchmark_alert_model
from utils.finetuning_functions import NewsAlertFineTuner
from transformers import AutoTokenizer, AutoModelForCausalLM



/opt/conda/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Data

In this section we will create the dataset that will be used for fine-tuning the LLM that we will use for the alert system. We use the dataset created from the Full_Dataset.py pipeline, that contains the articles, image_cpations, NER and SA outputs. 

**Golden dataset**
To create the golden dataset the model Gemini-3.1-Flash-Lite-preview was used. The dataset has the following information:
- Total processed articles: 1455

**Alert Generation Summary** We can see that the dataset is correctly balanced. This ensures that the fine-tuning will not force the model to over-alert or under-alert. We can also see that we havve a balanced number or alerts with negative sentiment, so the model won't alert automatically when it detects a negative sentiment.
- Articles with Alerts: 696 (47.8%)
- Articles without Alerts: 759 (52.2%)

--- Alert Levels ---

- info        624
- critical     72

--- Alert Categories ---

category
reputation_risk    : 231
geopolitical        :219
operational_risk    :120
legal_compliance     :42
esg_controversy      :35
executive_risk       :34
financial_market     :13
brand_safety          :2

- Filas con sentimiento negativo y con alerta: 695
- Filas con sentimiento negativo y sin alerta: 713

In [2]:
df = pd.read_csv("../data/clean_dataset.csv", sep=";", nrows=10000)
df.head(5)

,article,image_caption,NER,SA
0,Morning Mix‘She loved being a mother’: Pregnan...,Candace Pickens and her son Zachaeus,"[('Morning', 'O'), ('Mix‘She', 'O'), ('loved',...",negative
1,LocalGrand jury rejects criminal charges in de...,Robert Ethan Saylor is seen here in this famil...,"[('LocalGrand', 'O'), ('jury', 'O'), ('rejects...",negative
2,Arts and EntertainmentWhere to watch the 2016 ...,Sen Rand Paul and his wife Kelley,"[('Arts', 'O'), ('and', 'O'), ('EntertainmentW...",negative
3,"AfricaStricken with Ebola, a family runs away ...",UN reports Sierra Leone lagging in Ebola fight,"[('AfricaStricken', 'O'), ('with', 'O'), ('Ebo...",negative
4,EntertainmentOctober: 2011’s top buzz acts ret...,James Blake at Rock Roll Hotel in May,"[('EntertainmentOctober:', 'O'), ('2011’s', 'O...",negative


In [3]:
class AlertOutput(BaseModel):
    is_alert: bool = Field(..., description="Whether this news requires an alert.")
    alert_level: Optional[Literal["critical", "info"]] = Field(
        None, description="Critical: Immediate action. Info: Save for daily report."
    )
    category: Optional[Literal[
        "brand_safety",     
        "legal_compliance", 
        "operational_risk",  
        "financial_market",  
        "geopolitical",      
        "esg_controversy",   
        "executive_risk",
        "reputation_risk"     
    ]] = Field(None, description="The primary domain of the news.")
    reason: Optional[str] = Field(None, description="Short summary of why this was flagged.")

API_KEY = "GEMINI_API_KEY_(no subir, la mia la voy a ir borrando)"
client = genai.Client(api_key=API_KEY)

def generate_alert(article: str, image_caption: str, ner: list, sa: str):
    prompt = f"""
    Analyze the following news elements and determine if an alert should be triggered.
    ARTICLE: {article}
    IMAGE_CAPTION: {image_caption}
    NER (Entities): {ner}
    SENTIMENT_ANALYSIS: {sa}
    Return the analysis strictly in JSON format.
    """
    response = client.models.generate_content(
        model='models/gemini-3.1-flash-lite-preview',
        contents=prompt,
        config={
            'response_mime_type': 'application/json',
            'response_schema': AlertOutput,
            'thinking_config': {'include_thoughts': False} 
        }
    )
    return response.text

In [4]:
ADD_NEWS_ALERTS = False
MAX_REQUESTS_PER_DAY = 490 
requests_made = 0

START_INDEX = 1600
END_INDEX = 2100
GOLDEN_CSV_PATH = "../data/golden_fine_tuning_alerts.csv"

results = []
if ADD_NEWS_ALERTS:
    for index, row in df.iloc[START_INDEX:END_INDEX].iterrows():
        if requests_made >= MAX_REQUESTS_PER_DAY:
            print("Reached the maximum allowed requests for today.")
            break
            
        try:        
            alert_json_str = generate_alert(
                article=row.get('article', ''),
                image_caption=row.get('image_caption', ''),
                ner=normalize_ner_output(row.get('NER', '')),
                sa=row.get('SA', '')
            )
            
            alert = json.loads(alert_json_str)
            
            if not alert.get("is_alert"):
                final_output = "null"
            else:
                final_dict = {
                    "alert_level": alert.get("alert_level"),
                    "category": alert.get("category"),
                    "reason": alert.get("reason")
                }
                final_output = json.dumps(final_dict)
                
            results.append({
                "original_index": index,
                "article": row.get('article', ''),
                "image_caption": row.get('image_caption', ''),
                "NER": row.get('NER', ''),
                "SA": row.get('SA', ''),
                "Alert": final_output
            })
            
            requests_made += 1
            print(f"Processed request {requests_made}/{MAX_REQUESTS_PER_DAY} (DataFrame Index: {index})")
            time.sleep(4)
            
        except Exception as e:
            print(f"Error processing row {index}: {e}")
            time.sleep(10)

In [5]:
if ADD_NEWS_ALERTS:
    if len(results) > 0:
        fine_tuning_dataset = pd.DataFrame(results)

        if 'NER' in fine_tuning_dataset.columns:
            fine_tuning_dataset['NER'] = fine_tuning_dataset['NER'].apply(normalize_ner_output)
            
        write_header = not os.path.exists(GOLDEN_CSV_PATH)
        
        fine_tuning_dataset.to_csv(
            GOLDEN_CSV_PATH, 
            mode='a', 
            index=False, 
            header=write_header,
            sep=";"
        )
        print(f"Successfully appended {len(results)} rows to {GOLDEN_CSV_PATH}.")
        display(fine_tuning_dataset.head())
    else:
        print("No results to save.")

In [6]:
fine_tuning_dataset = pd.read_csv(GOLDEN_CSV_PATH, sep=";")
print(f"Total processed articles: {len(fine_tuning_dataset)}")

sa_norm = fine_tuning_dataset["SA"].astype(str).str.strip().str.lower()
alert_norm = fine_tuning_dataset["Alert"].astype(str).str.strip().str.lower()

is_no_alert = fine_tuning_dataset["Alert"].isna() | alert_norm.isin({"", "null", "none", "nan"})
is_alert = ~is_no_alert

null_alerts = int(is_no_alert.sum())
valid_alerts = int(is_alert.sum())

print(f"\n--- Alert Generation Summary ---")
print(f"Articles with Alerts: {valid_alerts} ({(valid_alerts/len(fine_tuning_dataset))*100:.1f}%)")
print(f"Articles without Alerts: {null_alerts} ({(null_alerts/len(fine_tuning_dataset))*100:.1f}%)")

if valid_alerts > 0:
    def parse_alert_json(alert_str):
        if pd.isna(alert_str):
            return None
        if isinstance(alert_str, str) and alert_str.strip().lower() in {"", "null", "none", "nan"}:
            return None
        try:
            return json.loads(alert_str)
        except Exception:
            return None
            
    parsed_alerts = fine_tuning_dataset["Alert"].apply(parse_alert_json).dropna()
    alerts_expanded = pd.json_normalize(parsed_alerts)
    print(f"\n--- Alert Levels ---")
    print(alerts_expanded["alert_level"].value_counts().to_string())   
    print(f"\n--- Alert Categories ---")
    print(alerts_expanded["category"].value_counts().to_string())

is_negative = sa_norm.eq("negative")
print("Filas con sentimiento negativo y con alerta:", int((is_negative & is_alert).sum()))
print("Filas con sentimiento negativo y sin alerta:", int((is_negative & is_no_alert).sum()))

Total processed articles: 1455

--- Alert Generation Summary ---
Articles with Alerts: 696 (47.8%)
Articles without Alerts: 759 (52.2%)

--- Alert Levels ---
alert_level
info        624
critical     72

--- Alert Categories ---
category
reputation_risk     231
geopolitical        219
operational_risk    120
legal_compliance     42
esg_controversy      35
executive_risk       34
financial_market     13
brand_safety          2
Filas con sentimiento negativo y con alerta: 695
Filas con sentimiento negativo y sin alerta: 713


# Fine-Tuning

In this section the fine-tuning of the model will be done using the generated data.



In [7]:

MODEL_NAME  = "Qwen/Qwen2.5-7B-Instruct"          
OUTPUT_DIR  = "../models/alert_finetuned"      
MERGED_DIR  = "../models/alert_finetuned_merged" 

NUM_EPOCHS  = 2
BATCH_SIZE  = 4     
LEARNING_RATE = 1e-4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUTPUT_CSV_PATH = "../data/golden_fine_tuning_alerts.csv"
df = pd.read_csv(GOLDEN_CSV_PATH, sep=";")
RESULTS_DIR = "../results/alert_finetuning"

TRAIN_DF_PATH = "../data/alert_finetuning_train.csv"
TEST_DF_PATH = "../data/alert_finetuning_test.csv"
VAL_DF_PATH = "../data/alert_finetuning_val.csv"


In [8]:
fine_tuner = NewsAlertFineTuner(model_name=MODEL_NAME)

# We have save the splits datasets for reproducibility
if not (os.path.exists(TRAIN_DF_PATH) and os.path.exists(VAL_DF_PATH) and os.path.exists(TEST_DF_PATH)):
    train_df, val_df, test_df = fine_tuner.load_and_split_data(
        csv_path=OUTPUT_CSV_PATH,
        train_ratio=0.8,
        val_ratio=0.1,
        test_ratio=0.1,
    )

    train_df.to_csv(TRAIN_DF_PATH, index=False, sep=";")
    val_df.to_csv(VAL_DF_PATH, index=False, sep=";")
    test_df.to_csv(TEST_DF_PATH, index=False, sep=";")

In [9]:

train_df = pd.read_csv(TRAIN_DF_PATH, sep=";")
val_df = pd.read_csv(VAL_DF_PATH, sep=";")
fine_tuner.load_model_and_tokenizer()
fine_tuner.setup_lora()

fine_tuner.fine_tune(
    train_prompts=fine_tuner.format_prompts(train_df, is_training=True),
    val_prompts=fine_tuner.format_prompts(val_df, is_training=True),
    output_dir=OUTPUT_DIR,
    num_epochs=NUM_EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
)


Loading Qwen/Qwen2.5-7B-Instruct with 4-bit quantization ...


Loading checkpoint shards: 100%|██████████| 4/4 [00:15<00:00,  3.88s/it]


Model loaded.
LoRA applied — trainable: 40,370,176 / 4,393,342,464 (0.92%)
Formatted 1164 prompts
Formatted 145 prompts

Starting fine-tuning — 1164 train / 145 val examples
Tokenizing with label masking ...
Label masking — answer tokens: 26,915 / 1,900,323 (1.4%)
Label masking — answer tokens: 3,110 / 242,203 (1.3%)


Epoch,Training Loss,Validation Loss
1,0.292100,nan
2,0.242600,nan


'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /Qwen/Qwen2.5-7B-Instruct/resolve/main/config.json (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x70c7eb514ad0>: Failed to establish a new connection: [Errno 101] Network is unreachable'))"), '(Request ID: 11a6e28c-f056-408a-90b4-41af3a4747fd)')' thrown while requesting HEAD https://huggingface.co/Qwen/Qwen2.5-7B-Instruct/resolve/main/config.json
Retrying in 1s [Retry 1/5].


Fine-tuning complete. Checkpoints saved to ../models/alert_finetuned


In [10]:
# We merge LoRA weights with the base model 
print("Merging LoRA adapters into base weights ...")
fine_tuner.merge_and_save(MERGED_DIR) 


Merging LoRA adapters into base weights ...
Merging LoRA adapters ...


Merged model saved to ../models/alert_finetuned_merged


# Testing & Metrics

This section benchmarks the prediction quality and compares it with de base model


In [12]:
def load_model(model_id: str):
    """Load tokenizer + model from a local path or HuggingFace repo ID."""
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
        device_map="auto" if DEVICE == "cuda" else None,
    )
    
    model.to(DEVICE)
    model.eval()
    return tokenizer, model

repo_id = "beaotero05/qwen2-Alarm-json"

print(f"Loading fine-tuned model: {MERGED_DIR}")
ft_tokenizer, ft_model = load_model(MERGED_DIR)

print(f"Loading base model: {MODEL_NAME}")
base_tokenizer, base_model = load_model(MODEL_NAME)

Loading fine-tuned model: ../models/alert_finetuned_merged


Loading checkpoint shards: 100%|██████████| 2/2 [00:02<00:00,  1.39s/it]


Loading base model: Qwen/Qwen2.5-7B-Instruct


Loading checkpoint shards: 100%|██████████| 4/4 [00:14<00:00,  3.74s/it]


In [13]:
def summarise(name: str, result: dict):
    m  = result["metrics"]
    cm = result["classification_metrics"]
    print(f"\n{'─'*50}")
    print(f"  {name}")
    print(f"{'─'*50}")
    print(f"  Entries          : {m['n_entries']}")
    print(f"  Mean latency (s) : {m['mean_time_seconds']:.3f}")
    if cm:
        print(f"  Accuracy         : {cm['accuracy']:.4f}")
        for label in ("alert", "no_alert"):
            r = cm["report"].get(label, {})
            if r:
                print(f"  {label:<10}  precision={r['precision']:.3f}  "
                      f"recall={r['recall']:.3f}  f1={r['f1-score']:.3f}")
                fm = result.get("field_metrics", {})
                if "alert_level" in fm:
                    print(f"  alert_level accuracy : {fm['alert_level']['accuracy']:.4f}  (n={fm['alert_level']['n']})")
                if "category" in fm:
                    print(f"  category accuracy    : {fm['category']['accuracy']:.4f}  (n={fm['category']['n']})")

In [16]:
benchmark_dataset = pd.read_csv(TEST_DF_PATH, sep=";")

print(f"Running benchmark on {len(benchmark_dataset)} examples ...\n")

from utils.finetuning_functions import generate_alert

ft_results = benchmark_alert_model(
    predict_fn=lambda entry: generate_alert(ft_model,   ft_tokenizer,   DEVICE, entry),
    dataset=benchmark_dataset,
    label_column="Alert",
)

base_results = benchmark_alert_model(
    predict_fn=lambda entry: generate_alert(base_model, base_tokenizer, DEVICE, entry),
    dataset=benchmark_dataset,
    label_column="Alert",
)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Running benchmark on 146 examples ...

Predicted  : 56 alert / 90 no_alert
Ground truth: 72 alert / 74 no_alert
Predicted  : 76 alert / 70 no_alert
Ground truth: 72 alert / 74 no_alert


In [17]:
summarise(f"Fine-tuned ({repo_id})", ft_results)
summarise(f"Base ({MODEL_NAME})", base_results)


──────────────────────────────────────────────────
  Fine-tuned (beaotero05/qwen2-Alarm-json)
──────────────────────────────────────────────────
  Entries          : 146
  Mean latency (s) : 2.330
  Accuracy         : 0.8082
  alert       precision=0.893  recall=0.694  f1=0.781
  alert_level accuracy : 0.3611  (n=72)
  category accuracy    : 0.6111  (n=72)
  no_alert    precision=0.756  recall=0.919  f1=0.829
  alert_level accuracy : 0.3611  (n=72)
  category accuracy    : 0.6111  (n=72)

──────────────────────────────────────────────────
  Base (Qwen/Qwen2.5-7B-Instruct)
──────────────────────────────────────────────────
  Entries          : 146
  Mean latency (s) : 1.732
  Accuracy         : 0.7945
  alert       precision=0.776  recall=0.819  f1=0.797
  alert_level accuracy : 0.4444  (n=72)
  category accuracy    : 0.5833  (n=72)
  no_alert    precision=0.814  recall=0.770  f1=0.792
  alert_level accuracy : 0.4444  (n=72)
  category accuracy    : 0.5833  (n=72)


## Upload the final tuned model to HF

We uploaded the fine-tuned model to Hugging Face in order to access it easily. It is public so it can be used by everyone.

In [ ]:
COMMITED = False
repo_id = "beaotero05/qwen2-Alarm-json"

if COMMITED:
    model = AutoModelForCausalLM.from_pretrained(
        MERGED_DIR,
        dtype=torch.float16,
        device_map="auto"
    )
    tokenizer = AutoTokenizer.from_pretrained(MERGED_DIR)

    model.push_to_hub(repo_id, safe_serialization=True)
    tokenizer.push_to_hub(repo_id)


## Ablation test

In this section, we evaluated the fine-tuned alert model by removing one input source at a time: image caption, NER, and sentiment analysis.  
This lets us measure how much each modality contributes to the final alert prediction and compare the performance against the full-input baseline.


In [ ]:
from utils.finetuning_functions import generate_alert
df_test = pd.read_csv("../data/alert_finetuning_full_benchmark.csv", sep=";")


In [ ]:


model = AutoModelForCausalLM.from_pretrained(
    repo_id,
    dtype=torch.float16,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(repo_id)



Loading checkpoint shards: 100%|██████████| 2/2 [00:02<00:00,  1.29s/it]


In [ ]:
# We evalute the fine-tuned model without providing the image caption
df= df_test.copy()
df["image_caption"] = ""
ft_results_without_caption = benchmark_alert_model(
    predict_fn=lambda entry: generate_alert(model, tokenizer,  DEVICE, entry),
    dataset=df,
    label_column="Alert",
)
summarise(f"Fine-tuned (without caption)", ft_results_without_caption)

Predicted  : 51 alert / 95 no_alert
Ground truth: 72 alert / 74 no_alert

──────────────────────────────────────────────────
  Fine-tuned (without caption)
──────────────────────────────────────────────────
  Entries          : 146
  Mean latency (s) : 1.885
  Accuracy         : 0.7877
  alert       precision=0.902  recall=0.639  f1=0.748
  alert_level accuracy : 0.3333  (n=72)
  category accuracy    : 0.5972  (n=72)
  no_alert    precision=0.726  recall=0.932  f1=0.817
  alert_level accuracy : 0.3333  (n=72)
  category accuracy    : 0.5972  (n=72)


In [ ]:
# We evalute the fine-tuned model without providing the NER entities
df = df_test.copy()
df["NER"] = ""
ft_results_without_ner = benchmark_alert_model(
    predict_fn=lambda entry: generate_alert(model, tokenizer,  DEVICE, entry),
    dataset=df,
    label_column="Alert",
)
summarise(f"Fine-tuned (without NER)", ft_results_without_ner)

Predicted  : 49 alert / 97 no_alert
Ground truth: 72 alert / 74 no_alert

──────────────────────────────────────────────────
  Fine-tuned (without NER)
──────────────────────────────────────────────────
  Entries          : 146
  Mean latency (s) : 1.850
  Accuracy         : 0.8014
  alert       precision=0.939  recall=0.639  f1=0.760
  alert_level accuracy : 0.3472  (n=72)
  category accuracy    : 0.6528  (n=72)
  no_alert    precision=0.732  recall=0.959  f1=0.830
  alert_level accuracy : 0.3472  (n=72)
  category accuracy    : 0.6528  (n=72)


In [ ]:
# We evalute the fine-tuned model without providing the SA 
df_test = pd.read_csv("../data/alert_finetuning_test.csv", sep=";")
df= df_test.copy()
df["SA"] = ""
df.head()
ft_results_without_sa = benchmark_alert_model(
    predict_fn=lambda entry: generate_alert(model, tokenizer,  DEVICE, entry),
    dataset=df,
    label_column="Alert",
)
summarise(f"Fine-tuned (without SA)", ft_results_without_sa)

Predicted  : 33 alert / 113 no_alert
Ground truth: 72 alert / 74 no_alert

──────────────────────────────────────────────────
  Fine-tuned (without SA)
──────────────────────────────────────────────────
  Entries          : 146
  Mean latency (s) : 1.633
  Accuracy         : 0.7055
  alert       precision=0.939  recall=0.431  f1=0.590
  alert_level accuracy : 0.1806  (n=72)
  category accuracy    : 0.5694  (n=72)
  no_alert    precision=0.637  recall=0.973  f1=0.770
  alert_level accuracy : 0.1806  (n=72)
  category accuracy    : 0.5694  (n=72)
